In [ ]:
#!pip install torch torchvision rasterio transformers segmentation-models-pytorch numpy tqdm scikit-learn

In [3]:
import json
import os
import os
import rasterio
import numpy as np
from tqdm import tqdm
import torch



# ==========================================
# 1. CARICAMENTO CONFIGURAZIONE DA JSON
# ==========================================

# Carica il file config.json
with open('config.json', 'r') as f:
    json_config = json.load(f)

# Mappa il JSON nella struttura CONFIG piatta usata dallo script
CONFIG = {
    # Percorsi
    "DATA_DIR": json_config["paths"]['input_dir'],
    # Nota: IMAGES_DIR e MASKS_DIR sono usati internamente da DATA_DIR nello script originale,
    # ma se vuoi essere specifico puoi usarli nella classe Dataset.
    # Per ora lo script originale si aspetta che dentro DATA_DIR ci siano "images" e "masks".
    
    # Parametri Modello e Dati
    "MODEL_NAME": json_config["model_config"]["model_name"],
    "NUM_CLASSES": json_config["data_specs"]["num_classes"],
    "IN_CHANNELS": json_config["data_specs"]["num_channels"],
    "IMG_SIZE": json_config["data_specs"]["img_size"],
    
    # Iperparametri di Training (Ottimizzati per 4GB VRAM)
    "BATCH_SIZE": json_config["training_params"]["batch_size"],
    "GRAD_ACCUM_STEPS": json_config["training_params"]["gradient_accumulation_steps"],
    "LEARNING_RATE": json_config["training_params"]["learning_rate"],
    "EPOCHS": json_config["training_params"]["num_epochs"],
    "NUM_WORKERS": json_config["training_params"]["num_workers"],
    
    # Hardware
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    
    # Normalizzazione (Statistiche calcolate)
    "MEANS": json_config["data_specs"]["normalization"]["means"],
    "STDS": json_config["data_specs"]["normalization"]["stds"]
}

print(f"Configurazione caricata per {CONFIG['MODEL_NAME']}")
print(f"Device: {CONFIG['DEVICE']} | Batch Reale: {CONFIG['BATCH_SIZE'] * CONFIG['GRAD_ACCUM_STEPS']}")

Configurazione caricata per ibm-nasa-geospatial/Prithvi-EO-2.0-100M-TL
Device: cuda | Batch Reale: 16


# calcola deviazioni std e mean per banda

--- Calcolo Statistiche Dataset in: /export/mimmo/output/images ---
Trovate 33018 immagini. Inizio calcolo...


  1%|          | 280/33018 [00:02<04:31, 120.57it/s]

100%|██████████| 33018/33018 [04:34<00:00, 120.15it/s]


RISULTATI DA COPIARE IN CONFIG:
"MEANS": [867.7125, 1231.0215, 1543.6707, 2885.9093, 1923.9377, 2509.7022, 2746.4171, 2936.5452, 3066.0805, 2402.2556],
"STDS":  [908.1966, 918.6142, 1087.9514, 1109.6611, 1092.8631, 1027.6976, 1069.1535, 1077.6171, 1258.3124, 1190.5524]

Note:
- Numero canali rilevati: 10
- Pixel totali analizzati: 2163867648
Incolla le due righe sopra nel dizionario CONFIG in train_sentinel.py


In [ ]:
import os
import random
import numpy as np
import rasterio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import AutoModel, AutoConfig
from tqdm import tqdm
import torch.nn.functional as F


In [ ]:


# ==========================================
# 2. DATASET CUSTOM (Gestione 10 Bande)
# ==========================================
class SentinelDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "images")
        self.mask_dir = os.path.join(root_dir, "masks")
        
        # Carica la lista dei file (presuppone file .tif)
        all_files = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.tif')])
        
        # Split semplice 80/20
        split_idx = int(len(all_files) * 0.8)
        if split == "train":
            self.files = all_files[:split_idx]
        else:
            self.files = all_files[split_idx:]
            
        self.means = np.array(CONFIG["MEANS"]).reshape(10, 1, 1)
        self.stds = np.array(CONFIG["STDS"]).reshape(10, 1, 1)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]
        
        # 1. Caricamento Immagine (10 Canali)
        img_path = os.path.join(self.img_dir, img_name)
        with rasterio.open(img_path) as src:
            image = src.read() # (Channels, H, W) -> float32 o uint16
            
        # 2. Caricamento Maschera
        mask_path = os.path.join(self.mask_dir, img_name) # Assicura che i nomi corrispondano
        with rasterio.open(mask_path) as src:
            mask = src.read(1) # (H, W) -> int
            
        # 3. Normalizzazione (Manuale per gestire 10 canali)
        image = image.astype(np.float32)
        image = (image - self.means) / (self.stds + 1e-6)
        
        # 4. Data Augmentation Semplice (Flip)
        if random.random() > 0.5:
            image = np.flip(image, axis=2).copy() # Horizontal flip
            mask = np.flip(mask, axis=1).copy()
            
        # Conversione in Tensor
        return torch.from_numpy(image), torch.from_numpy(mask).long()

# ==========================================
# 3. MODELLO (Prithvi Backbone + Decoder)
# ==========================================
class PrithviSegmentationModel(nn.Module):
    def __init__(self, model_name, num_classes, in_channels=10):
        super().__init__()
        
        # Carica la configurazione e il backbone ViT
        self.config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name)
        
        # --- HACK CRITICO: Adattamento da 6 a 10 Canali ---
        # Prithvi ha un PatchEmbed layer che si aspetta 6 canali.
        # Dobbiamo espanderlo a 10.
        old_patch_embed = self.backbone.patch_embed.proj # Conv2d(6, embed_dim, ...)
        new_in_channels = in_channels
        
        # Crea nuovo layer con 10 canali
        new_patch_embed = nn.Conv2d(
            in_channels=new_in_channels,
            out_channels=old_patch_embed.out_channels,
            kernel_size=old_patch_embed.kernel_size,
            stride=old_patch_embed.stride,
            padding=old_patch_embed.padding
        )
        
        # Inizializzazione dei pesi (Weight Inflation)
        with torch.no_grad():
            # Copia i pesi originali per i primi 6 canali
            new_patch_embed.weight[:, :6, :, :] = old_patch_embed.weight
            # Per i canali extra (7-10), usiamo la media dei canali originali per iniziare bene
            mean_weight = torch.mean(old_patch_embed.weight, dim=1, keepdim=True)
            for i in range(6, new_in_channels):
                new_patch_embed.weight[:, i:i+1, :, :] = mean_weight
                
            new_patch_embed.bias = old_patch_embed.bias
            
        self.backbone.patch_embed.proj = new_patch_embed
        # --------------------------------------------------
        
        # Decoder Semplice (U-Net style upsampling)
        # Nota: Un decoder reale sarebbe più complesso (es. UPerHead), 
        # ma questo basta per far funzionare il training su 4GB.
        embed_dim = self.config.embed_dim # Es. 768 per base, meno per 100M
        
        self.decoder = nn.Sequential(
            nn.Conv2d(embed_dim, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )

    def forward(self, x):
        # x shape: (B, 10, 224, 224)
        
        # Passaggio nel Backbone (ViT)
        outputs = self.backbone(pixel_values=x)
        
        # L'output del ViT è (B, Num_Patches, Embed_Dim)
        # Prithvi usa anche token temporali, prendiamo l'ultimo hidden state
        last_hidden_state = outputs.last_hidden_state 
        
        # Rimuovi il cls_token se presente (dipende dalla versione specifica di Prithvi)
        # Solitamente per Prithvi: (B, Sequence_Length, Embed_Dim)
        # Dobbiamo fare un "reshape" per tornare a (B, Embed_Dim, H_feat, W_feat)
        
        B, N, C = last_hidden_state.shape
        H_feat = W_feat = int(np.sqrt(N)) # Assumendo immagine quadrata e patch 16x16
        
        # Reshape da Sequenza a Immagine 2D
        features = last_hidden_state.permute(0, 2, 1).view(B, C, H_feat, W_feat)
        
        # Upsampling alla dimensione originale (224x224)
        features = F.interpolate(features, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        
        # Classification Head
        logits = self.decoder(features)
        
        return logits

# ==========================================
# 4. LOSS FUNCTION (Focal + Dice per Sbilanciamento)
# ==========================================
class FocalDiceLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight)
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        # Dice Loss approssimata
        # (Implementazione semplificata per brevità, focal è spesso sufficiente)
        return focal_loss

# ==========================================
# 5. TRAINING LOOP (Ottimizzato per 4GB VRAM)
# ==========================================
def train():
    print(f"--- Avvio Training su {CONFIG['DEVICE']} ---")
    
    # 1. Dataset & Dataloader
    train_dataset = SentinelDataset(CONFIG["DATA_DIR"], split="train")
    val_dataset = SentinelDataset(CONFIG["DATA_DIR"], split="val")
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, num_workers=CONFIG["NUM_WORKERS"])
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=CONFIG["NUM_WORKERS"])
    
    # 2. Model Setup
    model = PrithviSegmentationModel(CONFIG["MODEL_NAME"], CONFIG["NUM_CLASSES"], CONFIG["IN_CHANNELS"])
    model.to(CONFIG["DEVICE"])
    
    # Gradient Checkpointing (SALVA MEMORIA VRAM)
    model.backbone.gradient_checkpointing_enable()
    
    # 3. Optimizer & Loss
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LEARNING_RATE"])
    
    # Pesi per le classi (opzionale: calcola l'inverso della frequenza se hai i dati)
    # class_weights = torch.tensor([0.1, 1.0, ...]).to(CONFIG["DEVICE"])
    criterion = FocalDiceLoss() 
    
    # Mixed Precision Scaler
    scaler = GradScaler()
    
    # 4. Loop Epoche
    for epoch in range(CONFIG["EPOCHS"]):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}")
        
        optimizer.zero_grad()
        
        for step, (images, masks) in enumerate(progress_bar):
            images = images.to(CONFIG["DEVICE"])
            masks = masks.to(CONFIG["DEVICE"])
            
            # --- Mixed Precision Context ---
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
                loss = loss / CONFIG["GRAD_ACCUM_STEPS"] # Normalizza loss
            
            # Backward
            scaler.scale(loss).backward()
            
            # --- Gradient Accumulation ---
            if (step + 1) % CONFIG["GRAD_ACCUM_STEPS"] == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            epoch_loss += loss.item() * CONFIG["GRAD_ACCUM_STEPS"]
            progress_bar.set_postfix({"loss": loss.item() * CONFIG["GRAD_ACCUM_STEPS"]})
            
        # Validazione base
        print(f"Epoch {epoch+1} Train Loss: {epoch_loss / len(train_loader):.4f}")
        
        # Salva checkpoint
        torch.save(model.state_dict(), f"prithvi_finetuned_epoch_{epoch}.pth")

if __name__ == "__main__":
    # Crea cartelle dummy se non esistono per evitare crash immediati
    os.makedirs(os.path.join(CONFIG["DATA_DIR"], "images"), exist_ok=True)
    os.makedirs(os.path.join(CONFIG["DATA_DIR"], "masks"), exist_ok=True)
    
    train()